# 2026-09-01 — 機能A のフォールバックと機能B の初回実測

APIキーが 8/31 に届いたので、これまで動かせなかった **LLM を呼ぶ部分**を初めて回した日の記録。
問いは2つだけ。

1. **機能A** — 近傍分類が自信を持てなかった行を LLM に回すと、実際に正しくなるのか
2. **機能B** — 統計モデルに先に解かせ、その予測を証拠として LLM に最終判断させると、統計モデル単体より良くなるのか

用語: **MAE**（平均絶対誤差 = 予測が平均で何万円ずれるか。小さいほど良い）、
**accuracy**（正解率）、**アブレーション**（部品を1つ外して効いていたか確かめる実験）。

詳細は [`docs/2026-09-01-feature-fallback.md`](../docs/2026-09-01-feature-fallback.md) と
[`docs/2026-09-01-llm-predictor.md`](../docs/2026-09-01-llm-predictor.md)。
このノートは `results/*.csv` を読むだけなので **API は呼ばない（無料・再実行可）**。

In [4]:
import sys
import pandas as pd, numpy as np
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))          # unfold を import するため
R = ROOT / "results"
read = lambda n: pd.read_csv(R / n, encoding="utf-8-sig")
pd.set_option("display.width", 200)

cost = read("llm_cost.csv")
print(f"この日までの API 費用の累計: ${cost['費用_usd'].sum():.2f}"
      f"（{len(cost):,} 回の呼び出し）")

この日までの API 費用の累計: $31.77（3,128 回の呼び出し）


## 1. 機能A のフォールバック — 成立した

確信度の低い順に何割を LLM に回すか（`escalate_rate`）を振って、
タイトル列から作ったグレード名が正規表現版とどれだけ一致するかを見る。
宣言した10値は実データの 90.5% しかカバーしないので、
**「宣言値に限った accuracy」が主指標**（残り 9.5% は正解になりようがない）。

In [2]:
fb = read("feature_fallback.csv")
print(fb.round(3).to_string(index=False))

base = fb.loc[0, "宣言値に限った accuracy"]
for _, r in fb.iloc[1:].iterrows():
    n = int(r["実際に回した行"])
    print(f"{r['エスカレーション率']:.0%} 回す → 宣言値内 accuracy "
          f"{r['宣言値に限った accuracy']:.3f}（{r['宣言値に限った accuracy']-base:+.3f}） / "
          f"{n} 行 / ${r['費用_usd']:.3f}（1行 ${r['費用_usd']/n:.4f}）")
print("\n→ LLM に回した行のうち宣言値内のものは、どの率でも 100% 正解だった。")
print("→ 15% 回すだけで改善の 3 分の 2 が取れる。全行に投げる必要はない。")

 エスカレーション率  実際に回した行  accuracy  宣言値に限った accuracy  回した行の accuracy  回した行の宣言値内 accuracy  費用_usd
      0.00        0     0.790             0.873             NaN                 NaN   0.000
      0.05       20     0.832             0.920           0.950                 1.0   0.106
      0.15       60     0.852             0.942           0.917                 1.0   0.215
      0.30      120     0.885             0.978           0.842                 1.0   0.333
5% 回す → 宣言値内 accuracy 0.920（+0.047） / 20 行 / $0.106（1行 $0.0053）
15% 回す → 宣言値内 accuracy 0.942（+0.069） / 60 行 / $0.215（1行 $0.0036）
30% 回す → 宣言値内 accuracy 0.978（+0.105） / 120 行 / $0.333（1行 $0.0028）

→ LLM に回した行のうち宣言値内のものは、どの率でも 100% 正解だった。
→ 15% 回すだけで改善の 3 分の 2 が取れる。全行に投げる必要はない。


## 2. 機能B — 統計モデルに届かず。ただし弱い証拠を外すと有意に改善

各 fold の test から 60 行ずつ、計 300 行で採点。
**統計モデルと LLM をまったく同じ行で採点している**ので比較は成り立つ（全行の 12.21 とは直接比べられない)。

- `all` … 設計書どおり証拠を全部渡す（LightGBM・XGBoost・**近傍5件の中央値**）
- `trees` … 弱い証拠（近傍中央値・単体 MAE 30.86）を外したアブレーション

In [3]:
a, t = read("llm_predictor.csv"), read("llm_predictor_trees.csv")
tbl = pd.Series({
    "機能B（証拠 = 全部・設計書どおり）": a["MAE_機能B"].mean(),
    "機能B（証拠 = 木2つだけ）":          t["MAE_機能B"].mean(),
    "LightGBM（＝超えるべき線）":         a["MAE_LightGBM"].mean(),
    "XGBoost":                            a["MAE_XGBoost"].mean(),
    "近傍5件の価格中央値":                 a["MAE_近傍5件の中央値"].mean(),
}, name="MAE（万円）")
print(tbl.round(2).to_string(), "\n")

# 行ごとのブートストラップ（データを復元抽出して差の分布を見る）で有意性を確かめる
d  = read("llm_predictor_trees_rows.csv")
y, llm, lg = (d[c].to_numpy(float) for c in ("実際", "機能B", "LightGBM"))
llm_all = read("llm_predictor_rows.csv")["機能B"].to_numpy(float)
idx = np.random.default_rng(0).integers(0, len(y), size=(2000, len(y)))

for name, diff in [("trees − all（弱い証拠を外した効果）", np.abs(llm-y) - np.abs(llm_all-y)),
                   ("trees − LightGBM（S6 の受け入れ基準）", np.abs(llm-y) - np.abs(lg-y))]:
    bs = diff[idx].mean(axis=1)
    lo, hi = np.percentile(bs, [2.5, 97.5])
    print(f"{name}: {diff.mean():+.3f} 万円 / 95%区間 [{lo:+.3f}, {hi:+.3f}] / "
          f"良い確率 {(bs < 0).mean():.1%}"
          f"{'  ← 有意' if hi < 0 else '  ← 0 をまたぐ＝有意でない'}")

機能B（証拠 = 全部・設計書どおり）    13.59
機能B（証拠 = 木2つだけ）        12.85
LightGBM（＝超えるべき線）      12.99
XGBoost                13.56
近傍5件の価格中央値             30.86 

trees − all（弱い証拠を外した効果）: -0.744 万円 / 95%区間 [-1.153, -0.357] / 良い確率 100.0%  ← 有意
trees − LightGBM（S6 の受け入れ基準）: -0.139 万円 / 95%区間 [-1.012, +0.728] / 良い確率 61.3%  ← 0 をまたぐ＝有意でない


## 3. どの行で LLM を信じるか — 信号を探した

行ごとに LLM と LightGBM の良いほうを選べたら MAE 10.37（**オラクル**＝実現不可能な上限）。
LightGBM の 12.99 より 2.6 も良いので、見分ける信号があれば伸びしろは大きい。
LLM の自己申告 confidence を含む 7 通りを、上位 r% だけ LLM を採用して比べた。

In [4]:
sig = read("routing_signals.csv").set_index("Unnamed: 0").rename_axis(None)
print(sig.round(2).to_string(), "\n")

best = sig.stack().idxmin()
print(f"最良: 「{best[0]}」を {best[1]} 採用 → MAE {sig.stack().min():.2f}"
      f"（LightGBM 12.99 / オラクル 10.37）")
print("→ ただし信号も採用率も同じ 300 行の上で選んでいるので、この数字は楽観側に偏る。")
print("→ 検定すると最良でも 95%区間が 0 をまたぐ。「有望な候補が見えた」以上は言えない。")

                              0%    10%    20%    30%    50%    75%   100%
LLM の自己申告 confidence（高い順）  12.99  12.95  12.90  12.87  12.86  12.88  12.85
木2つの食い違い |LGBM−XGB|（大きい順）  12.99  12.72  12.84  12.87  12.92  12.96  12.85
木2つの食い違い（小さい順）             12.99  13.02  12.91  12.95  12.91  13.09  12.85
近傍1位の類似度（低い順）              12.99  13.07  13.13  13.08  13.08  12.92  12.85
近傍1位の類似度（高い順）              12.99  12.77  12.81  12.95  12.75  12.69  12.85
LLM が動かした量（大きい順）           12.99  12.91  12.85  12.78  12.82  12.80  12.85
LLM が動かした量（小さい順）           12.99  12.98  13.02  13.03  13.02  13.15  12.85 

最良: 「近傍1位の類似度（高い順）」を 75% 採用 → MAE 12.69（LightGBM 12.99 / オラクル 10.37）
→ ただし信号も採用率も同じ 300 行の上で選んでいるので、この数字は楽観側に偏る。
→ 検定すると最良でも 95%区間が 0 をまたぐ。「有望な候補が見えた」以上は言えない。


## 4. ここまでの中間まとめ（シエンタ段階）

1. **機能A のフォールバックは成功。** 回した行は全問正解、1行 $0.003〜0.005。
   信頼度ルーティング（PRD §6.3）の前提はこの課題では成立している。
2. **シエンタでは、証拠から弱いモデルを外すと有意に改善した。**
   近傍中央値を混ぜると LLM が引きずられ、MAE が 0.74 悪化する。
   > **【訂正・この日の後半】この発見は一般則ではなかった。**
   > 下の「6. アブレーションは再現しなかった」を参照。Craigslist では符号が逆になる。
3. **選別しても LightGBM を有意には超えていない。** シエンタでは受け入れ基準 S6 は未達。
4. **confidence は機能A では効き、機能B では効かない。**
   「テキストに答えが書いてある分類」と「数値証拠を重み付ける回帰」で
   LLM の得意不得意が分かれている可能性。
5. **まだ単一車種（シエンタ）でしか測っていない。**
   過去に何度も「単一車種では差が出ない」（PRD §2.2-b）を踏んでいるので、**次はここ**。

**→ 5 をこの日のうちに実行した。結論はひっくり返った。以下がその記録。**

---

# 後半 — 複数車種（Craigslist）で測り直したら結論がひっくり返った

ここから先は同じ日の後半。**実装は一切変えず、データだけ替えた。**

Craigslist は 42 メーカー・数千車種が混在し、`model` 列に「f-150 raptor」
「x5 3.0i awd」のような**車種そのものの情報**が入っている。
シエンタは全車が同じ車種なので、テキストに書けることが装備の羅列しかなかった。

測り方は完全に同じ。60,000行を5分割し、各 fold の test から 120 行ずつ、計 600 行を採点する。

## 5. 公平な比較線を1本足した

**ここが結果を読むうえでいちばん大事。**

証拠として渡す木モデルには、**わざとテキストを渡していない**（LLM がテキストを
読めた効果だけを取り出すため）。しかしそのままだと、機能B が勝っても
**「LLM が賢いから」なのか「単にテキストが効くから」なのか区別できない。**

そこで**テキストを文字 TF-IDF で木に入れた LightGBM**（＝リーダーボードの最良構成）を
比較線として並べた。証拠には渡さず、採点だけする。

In [5]:
veh   = read("llm_predictor_vehicles.csv")          # 証拠 = 全部
veh_t = read("llm_predictor_vehicles_trees.csv")    # 証拠 = 木2つだけ
veh_d = read("llm_predictor_vehicles_desc2000.csv") # + 自由記述（金額は伏字）
ref   = read("llm_predictor_vehicles_reference.csv")["MAE_参考_LGBM文字TFIDF"]

tbl = pd.Series({
    "近傍5件の価格中央値":                 veh["MAE_近傍5件の中央値"].mean(),
    "XGBoost（構造化列のみ）":             veh["MAE_XGBoost"].mean(),
    "LightGBM（構造化列のみ・証拠）":       veh["MAE_LightGBM"].mean(),
    "参考: LightGBM+文字TF-IDF（既知最良）": ref.mean(),
    "機能B（model のみ・証拠=木2つ）":      veh_t["MAE_機能B"].mean(),
    "機能B（model のみ・証拠=全部）":       veh["MAE_機能B"].mean(),
    "機能B（model+自由記述）":             veh_d["MAE_機能B"].mean(),
}, name="MAE（USD）")
print(tbl.round(2).to_string())

best = ref.mean()
print(f"\n既知の最良構成 {best:,.2f} に対して")
for label in ["機能B（model のみ・証拠=全部）", "機能B（model+自由記述）"]:
    print(f"  {label}: {tbl[label]:,.2f}（{(tbl[label]-best)/best:+.1%}）")
print("\n※ 比較線 2,591.58 は 60,000 行での既知の値 2,595 とほぼ一致した。設定は健全。")

近傍5件の価格中央値                     3431.01
XGBoost（構造化列のみ）                3118.10
LightGBM（構造化列のみ・証拠）            3103.03
参考: LightGBM+文字TF-IDF（既知最良）    2591.58
機能B（車種名のみ・証拠=木2つ）              2362.71
機能B（車種名のみ・証拠=全部）               2335.32
機能B（車種名+自由記述）                  1980.68

既知の最良構成 2,591.58 に対して
  機能B（車種名のみ・証拠=全部）: 2,335.32（-9.9%）
  機能B（車種名+自由記述）: 1,980.68（-23.6%）

※ 比較線 2,591.58 は 60,000 行での既知の値 2,595 とほぼ一致した。設定は健全。


In [6]:
from scipy import stats as st

# fold ごとに、比較線と機能B を突き合わせる（同じ fold・同じ行での対応比較）
fold = pd.DataFrame({
    "参考線":            ref.to_numpy(),
    "機能B(model)":      veh["MAE_機能B"].to_numpy(),
    "機能B(+自由記述)":   veh_d["MAE_機能B"].to_numpy(),
}, index=pd.Index(range(1, 6), name="fold"))
print(fold.round(0).to_string(), "\n")

for col in ["機能B(model)", "機能B(+自由記述)"]:
    t, p = st.ttest_rel(fold[col], fold["参考線"])
    wins = int((fold[col] < fold["参考線"]).sum())
    print(f"{col} vs 参考線: 差 {fold[col].mean()-fold['参考線'].mean():+,.1f} USD / "
          f"対応t検定 p={p:.3f} / 勝った fold {wins}/5"
          f"{'  ← 有意' if p < 0.05 else ''}")
print("\n→ シエンタでは 95%区間が 0 をまたいだが、ここでは両方とも有意。")
print("→ PRD の受け入れ基準 S6 は、複数車種データでは満たされた。")

         参考線  機能B(車種名)  機能B(+自由記述)
fold                              
1     2232.0    2157.0      2025.0
2     2777.0    2392.0      2100.0
3     3661.0    3360.0      2542.0
4     2200.0    2004.0      1572.0
5     2088.0    1764.0      1665.0 

機能B(車種名) vs 参考線: 差 -256.3 USD / 対応t検定 p=0.009 / 勝った fold 5/5  ← 有意
機能B(+自由記述) vs 参考線: 差 -610.9 USD / 対応t検定 p=0.016 / 勝った fold 5/5  ← 有意

→ シエンタでは 95%区間が 0 をまたいだが、ここでは両方とも有意。
→ PRD の受け入れ基準 S6 は、複数車種データでは満たされた。


## 6. アブレーションは再現しなかった【重要な訂正】

シエンタでは「**弱い証拠（近傍中央値）が LLM を汚染する**」ことが有意に示され、
そこから「証拠は選別して渡すべきだ」という一般則を引き出しかけていた（上の中間まとめ 2）。

**Craigslist では再現しなかった。それどころか符号が逆である。**

In [7]:
sien_a, sien_t = read("llm_predictor.csv"), read("llm_predictor_trees.csv")
cmp = pd.DataFrame({
    "シエンタ（万円）":    [sien_a["MAE_機能B"].mean(), sien_t["MAE_機能B"].mean()],
    "Craigslist（USD）": [veh["MAE_機能B"].mean(),   veh_t["MAE_機能B"].mean()],
}, index=["証拠 = 木2つ + 近傍中央値（設計書どおり）", "証拠 = 木2つのみ"])
print(cmp.round(2).to_string(), "\n")

for name, a_rows, t_rows in [
        ("シエンタ",   "llm_predictor_rows.csv",          "llm_predictor_trees_rows.csv"),
        ("Craigslist", "llm_predictor_vehicles_rows.csv", "llm_predictor_vehicles_trees_rows.csv")]:
    A, T = read(a_rows), read(t_rows)
    y = A["実際"].to_numpy(float)
    ea, et = np.abs(A["機能B"].to_numpy(float)-y), np.abs(T["機能B"].to_numpy(float)-y)
    idx = np.random.default_rng(0).integers(0, len(y), size=(2000, len(y)))
    bs = (et[idx] - ea[idx]).mean(axis=1)
    lo, hi = np.percentile(bs, [2.5, 97.5])
    print(f"{name:11} trees − all: {et.mean()-ea.mean():+8.2f} / "
          f"95%区間 [{lo:+8.2f}, {hi:+8.2f}]"
          f"{'  ← 有意' if hi < 0 else '  ← 0 をまたぐ＝有意でない'}")

                          シエンタ（万円）  Craigslist（USD）
証拠 = 木2つ + 近傍中央値（設計書どおり）     13.59          2335.32
証拠 = 木2つのみ                   12.85          2362.71 

シエンタ        trees − all:    -0.74 / 95%区間 [   -1.15,    -0.36]  ← 有意
Craigslist  trees − all:   +27.38 / 95%区間 [  -10.90,   +66.51]  ← 0 をまたぐ＝有意でない


In [8]:
# なぜ違ったのか。引きずられ具合ではなく、LLM の判断そのものの質が違った
for name, rows in [("シエンタ",   "llm_predictor_trees_rows.csv"),
                   ("Craigslist", "llm_predictor_vehicles_trees_rows.csv")]:
    d = read(rows)
    y, llm, lg = (d[c].to_numpy(float) for c in ("実際", "機能B", "LightGBM"))
    knn = read(rows.replace("_trees", ""))["近傍5件の中央値"].to_numpy(float)
    better = (np.abs(llm-y) < np.abs(lg-y)).mean()
    corr = np.corrcoef(llm-lg, knn-lg)[0, 1]
    print(f"{name:11} LLM が動かして良くなった行 {better:.1%} / "
          f"修正方向と近傍中央値の相関 {corr:+.2f}")

print("\n→ 引きずられ具合（相関）は同じ。違うのは LLM の判断の質。")
print("   シエンタでは修正がコイン投げなので、入力が増えるほど雑音が増える。")
print("   Craigslist では 3 分の 2 が当たるので、弱い証拠も適切に割り引ける。")
print("\n→ 正しい一般則: **証拠の選別は機能B が効かないときの対症療法であって、")
print("   効かせるための処方ではない。**")

シエンタ        LLM が動かして良くなった行 48.0% / 修正方向と近傍中央値の相関 +0.56
Craigslist  LLM が動かして良くなった行 67.5% / 修正方向と近傍中央値の相関 +0.64

→ 引きずられ具合（相関）は同じ。違うのは LLM の判断の質。
   シエンタでは修正がコイン投げなので、入力が増えるほど雑音が増える。
   Craigslist では 3 分の 2 が当たるので、弱い証拠も適切に割り引ける。

→ 正しい一般則: **証拠の選別は機能B が効かないときの対症療法であって、
   効かせるための処方ではない。**


## 7. 自由記述を足す — その前にリークを見つけた

本命の `description`（出品者が書いた自由記述・平均 2,288 字）を足したら、
MAE が 2,335 → **1,546** と 40% 改善した。**良すぎる。**

価格中央値 13,950 USD に対して MAE 1,546 は 11% の誤差で、実務の査定士より良い。
**改善幅が桁違いなら、精度が上がったのではなく問題が簡単になっている**と疑うのが鉄則。

調べたら、**description の 43.7% に価格そのものが書いてあった。**

> Just Lowered! 2008 Toyota Sienna LE 222,617 miles **$5,900** plus DMV Fees …

出品ページの本文なので当然である。**予測ではなく答えを読んでいた。**

In [9]:
leak = read("description_leak.csv")
print(leak.round(3).to_string(index=False), "\n")
print("読み方:")
print("  改善率が **記載あり側だけ極端に大きい** → 読み取り（リーク）")
print("  改善率が **両側でほぼ同じ**            → 本物の効果")
print("\n→ 伏字なしは記載あり側だけ 64.7% 改善（読み取りの兆候）。")
print("→ 伏字ありは 14.6% と 15.6% でほぼ同じ（本物の効果の兆候）。**伏字は効いた。**")

            構成   全600行  価格記載あり    記載なし  記載あり行数  記載なし行数  記載ありの改善率  記載なしの改善率
         説明文なし 2335.32 2035.29 2567.89     262     338     0.000     0.000
説明文あり・伏字なし【無効】 1545.86  718.65 2187.07     262     338     0.647     0.148
    説明文あり・伏字あり 1980.68 1738.89 2168.10     262     338     0.146     0.156 

読み方:
  改善率が **記載あり側だけ極端に大きい** → 読み取り（リーク）
  改善率が **両側でほぼ同じ**            → 本物の効果

→ 伏字なしは記載あり側だけ 64.7% 改善（読み取りの兆候）。
→ 伏字ありは 14.6% と 15.6% でほぼ同じ（本物の効果の兆候）。**伏字は効いた。**


### 塞ぎ方 — ライブラリの責務にした

これは中古車に限らない。**出品・求人・不動産など、自由記述に価格や条件が
書かれるデータでは常に起きる。**「利用者が気をつける」では必ず事故るので、
`unfold` 側で既定で塞ぐことにした。

In [10]:
from unfold.predictor import mask_amounts

for t in ["Just Lowered! 2008 Toyota Sienna LE 222,617 miles $5,900 plus DMV Fees",
          "Ford F-150 Raptor 5.7L V8, Tesla Model 3, asking 12500 dollars"]:
    print(t)
    print("  →", mask_amounts(t), "\n")

print("走行距離・年式も巻き込むが、構造化列として別に渡しているので損はない。")
print("F-150 / 5.7L / Model 3 のような車種の数字は 300 未満なので残る。")
print("\n完全ではない（「twelve thousand」のような綴りは抜ける）。")
print("**伏字だけに頼らず、測定のたびに上の表でリーク検査をすること。**")

Just Lowered! 2008 Toyota Sienna LE 222,617 miles $5,900 plus DMV Fees
  → Just Lowered! 〈数値〉 Toyota Sienna LE 〈数値〉 miles 〈金額〉 plus DMV Fees 

Ford F-150 Raptor 5.7L V8, Tesla Model 3, asking 12500 dollars
  → Ford F-150 Raptor 5.7L V8, Tesla Model 3, asking 〈金額〉 

走行距離・年式も巻き込むが、構造化列として別に渡しているので損はない。
F-150 / 5.7L / Model 3 のような車種の数字は 300 未満なので残る。

完全ではない（「twelve thousand」のような綴りは抜ける）。
**伏字だけに頼らず、測定のたびに上の表でリーク検査をすること。**


## 8. 事前スクリーニング — LLM を呼ぶ前に効くかを見積もる

シエンタでは負け、Craigslist では勝った。差は**データの性質**にあった。
だったら「まず全行に LLM を投げてみる」のは時間と費用の両方で高くつく。

`unfold.screen()` は **LLM を1回も呼ばずに**、そのデータでテキストが効くかを測る。
同じ LightGBM を2本、テキストを入れる/入れないだけ変えて交差検証し、その差を見る。

In [11]:
scr = read("screening.csv")
print(scr.to_string(index=False), "\n")
print("→ 3列とも実測どおりに判定した（シエンタ=負けた / model=勝った）。")
print("\n【副産物】description は 3.7%「効きにくい」と判定された。文字 TF-IDF は")
print("description から価格を取り出せないので、この判定自体は正しい。しかし LLM は")
print("`$5,900` を**数値として読める**ので、TF-IDF にできないことをやってのけた。")
print("\n→ **スクリーニングと LLM の実測が大きく食い違ったら、それはリークの兆候。**")
print("   偶然の副産物だが、実用的な検査になる。")

       データ      列    行数  単位  値の種類  平均文字数  テキスト無しMAE  テキスト有りMAE  テキスト寄与率     判定
      シエンタ 装備テキスト  5507  万円  4678   63.3      12.88      12.32   0.0434  効きにくい
Craigslist    車種名 20000 USD  4915   10.0    3407.48    2882.14   0.1542 試す価値あり
Craigslist    説明文 20000 USD 19900 2288.1    3407.48    3280.35   0.0373  効きにくい

 

→ 3列とも実測どおりに判定した（シエンタ=負けた / 車種名=勝った）。

【副産物】説明文は 3.7%「効きにくい」と判定された。文字 TF-IDF は
説明文から価格を取り出せないので、この判定自体は正しい。しかし LLM は
`$5,900` を**数値として読める**ので、TF-IDF にできないことをやってのけた。

→ **スクリーニングと LLM の実測が大きく食い違ったら、それはリークの兆候。**
   偶然の副産物だが、実用的な検査になる。


## 9. レイテンシ（P4 の残り半分）

PRD §6.3 は「精度・レイテンシ・費用の3つが同時に見えること」を要件にしている。
精度と費用は取れたので、応答時間を測った（キャッシュを切って実際に呼ぶ）。

In [12]:
lat = read("latency_vehicles.csv")
print(lat.to_string(index=False), "\n")
print("読み方:")
print(f"  1件の応答は中央値 {lat['1件の中央値_秒'].mean():.1f} 秒。**並列数を上げても変わらない**")
print("  （API 側の素の時間なので、こちらでは縮められない）")
print(f"  全体時間は並列数に反比例し、費用はほぼ一定"
      f"（${lat['費用_usd'].min():.3f}〜${lat['費用_usd'].max():.3f}）")
print("  → 急ぐなら並列数を上げるのが正解。レート制限に当たるまで")
print("\n  準備処理（統計モデルの fit + 近傍索引の構築）は 23.85 秒で、行数に対して1回だけ")
print("\n  バッチ処理なら問題ない（60,000 行を並列8で約17時間、並列32 なら約4時間）")
print("  **対話的な用途では 4.6 秒は長い。** 統計モデルだけなら 6 ミリ秒で返せるので、")
print("  そこで信頼度ルーティング（迷う行だけ LLM に回す）が効いてくる。")

 並列数  全体_秒  1行あたり_秒  1件の中央値_秒  1件のp90_秒  1件の最大_秒  費用_usd  エラー
   1 88.62    5.908      4.59      5.78    18.79  0.1368    0
   4 23.92    1.595      4.38      5.02     5.44  0.1332    0
   8 15.49    1.033      4.63      5.36     5.46  0.1346    0 

読み方:
  1件の応答は中央値 4.5 秒。**並列数を上げても変わらない**
  （API 側の素の時間なので、こちらでは縮められない）
  全体時間は並列数に反比例し、費用はほぼ一定（$0.133〜$0.137）
  → 急ぐなら並列数を上げるのが正解。レート制限に当たるまで

  準備処理（統計モデルの fit + 近傍索引の構築）は 23.85 秒で、行数に対して1回だけ

  バッチ処理なら問題ない（60,000 行を並列8で約17時間、並列32 なら約4時間）
  **対話的な用途では 4.6 秒は長い。** 統計モデルだけなら 6 ミリ秒で返せるので、
  そこで信頼度ルーティング（迷う行だけ LLM に回す）が効いてくる。


## 10. 結論（この日の最終版）

### 決着したこと

1. **機能B は成立する。ただし「非構造テキストが価格を左右するデータ」でのみ。**
   同じ実装が単一車種では負け、複数車種では有意に勝った（p=0.009〜0.016、5/5 fold）。
   **実装の良し悪しではなく適用対象の性質の問題**だった。
2. **到達点は既知の最良構成に対して −23.6%**（2,591.58 → 1,980.68）。
   PRD の受け入れ基準 S6 を満たした。
3. **機能A のフォールバックも成功。** 確信度の低い 15% を回すと
   宣言値内 accuracy 0.873 → 0.942、回した行は全問正解、1行 $0.0036。
4. **P4 完了。** 精度・費用・レイテンシの3つが揃った。

### 取り消したこと

- **「設計書の『複数モデルの出力を全部渡す』は間違い」は一般則ではなかった。**
  シエンタ固有で、Craigslist では符号が逆になった。
  正しくは「**証拠の選別は機能B が効かないときの対症療法**」。

### 新たに分かった危険

- **自由記述には目的変数が書いてあると思って設計する。** 出品データではむしろ普通。
  `mask_amounts()` を既定にしたが完全ではないので、**測定のたびに部分集合で検査する。**

### 東京で相談したいこと

1. **機能B のゴールを精度に置くか、来歴・説明可能性に置くか。**
   精度では複数車種で勝ったが、単一車種では勝てない。適用条件がはっきりした以上、
   「どこで使うべき道具か」を先に決めたほうが実装の優先順位が決まる。
2. **`unfold.screen()` の閾値 10% の根拠が2点しかない。**
   もっとデータセットを増やすか、閾値という形をやめるか。
3. **description を全行に渡すと 60,000 行で約 $700。** 信頼度ルーティングで絞る前提の設計に
   するか、そもそも安いモデル（Haiku）で足りるかを測るか。